1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰 수 초과 방지, 답변 생성 시간 단축
3. 임베딩해서 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [1]:
%pip install -qU  docx2txt langchain-community


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크가 가질 수 있는 토큰 수
    chunk_overlap=200, # 이전 청크와 현재 청크 겹치는 내용 양양
)

loader = Docx2txtLoader('./tax.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
%pip install -qU langchain-text-splitters


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 임베딩하기
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [8]:
# 벡터 DB
%pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 10.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 8.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 8.7 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 5.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 4.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 3.0 MB

In [3]:
from langchain_chroma import Chroma

database = Chroma.from_documents(
    documents=document_list, 
    embedding=embedding, 
    persist_directory="./chroma",
    collection_name='chroma-tax'   
)

In [10]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=5)

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [15]:
prompt= f"""[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다
- [Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrieved_docs}

Question: {query}
"""

In [16]:
ai_message = llm.invoke(prompt)

In [17]:
ai_message.content

'직장인의 소득세 계산은 여러 요소에 따라 달라지며, 기본적인 계산은 다음과 같은 과정을 포함합니다:\n\n1. **총급여액 추정**: 연봉 5천만 원이라고 가정합니다. \n2. **근로소득공제 적용**: [Context] 문서에는 명시되지 않았지만, 일반적으로 근로소득공제는 연봉에 따라 일정 비율로 적용됩니다. 이것은 소득세를 계산할 때 공제되는 금액입니다.\n3. **과세표준 계산**: 총급여액에서 근로소득공제를 뺀 금액이 과세표준이 됩니다.\n4. **세율 적용**: 과세표준에 소득세법에 명시된 세율을 적용하여 세금을 계산합니다. 한국의 소득세는 구간별로 누진세율이 적용되며, 일반적으로 아래와 같은 세율이 적용됩니다 (이는 법 개정에 따라 변경될 수 있습니다):\n   - 1,200만원 이하: 6%\n   - 1,200만원 초과 4,600만원 이하: 15%\n   - (이외 구간에 대해서는 추가적인 세율이 존재합니다)\n5. **세액 공제**: 계산된 세액에서 각종 세액 공제를 적용할 수 있습니다.\n\n구체적인 금액은 개인의 소득상황, 공제 항목, 세액공제 등에 따라 달라질 수 있으므로, 근로소득간이세액표나 세무 전문가의 상담을 통해 정확한 금액을 파악할 수 있습니다. 다만, 일반적으로 말씀드리면, 5천만원의 연봉에서는 약 10~15% 범주 내의 소득세를 예상할 수 있습니다. 이를 통해 약 500만원에서 750만원 정도의 소득세가 부과될 가능성이 있습니다. 정확한 금액을 알아보기 위해 개인의 공제 항목 등을 고려한 구체적인 계산이 필요합니다.'

'직장인의 소득세 계산은 여러 요소에 따라 달라지며, 기본적인 계산은 다음과 같은 과정을 포함합니다:\n\n1. **총급여액 추정**: 연봉 5천만 원이라고 가정합니다. \n2. **근로소득공제 적용**: [Context] 문서에는 명시되지 않았지만, 일반적으로 근로소득공제는 연봉에 따라 일정 비율로 적용됩니다. 이것은 소득세를 계산할 때 공제되는 금액입니다.\n3. **과세표준 계산**: 총급여액에서 근로소득공제를 뺀 금액이 과세표준이 됩니다.\n4. **세율 적용**: 과세표준에 소득세법에 명시된 세율을 적용하여 세금을 계산합니다. 한국의 소득세는 구간별로 누진세율이 적용되며, 일반적으로 아래와 같은 세율이 적용됩니다 (이는 법 개정에 따라 변경될 수 있습니다):\n   - 1,200만원 이하: 6%\n   - 1,200만원 초과 4,600만원 이하: 15%\n   - (이외 구간에 대해서는 추가적인 세율이 존재합니다)\n5. **세액 공제**: 계산된 세액에서 각종 세액 공제를 적용할 수 있습니다.\n\n구체적인 금액은 개인의 소득상황, 공제 항목, 세액공제 등에 따라 달라질 수 있으므로, 근로소득간이세액표나 세무 전문가의 상담을 통해 정확한 금액을 파악할 수 있습니다. 다만, 일반적으로 말씀드리면, 5천만원의 연봉에서는 약 10~15% 범주 내의 소득세를 예상할 수 있습니다. 이를 통해 약 500만원에서 750만원 정도의 소득세가 부과될 가능성이 있습니다. 정확한 금액을 알아보기 위해 개인의 공제 항목 등을 고려한 구체적인 계산이 필요합니다.'

In [ ]:
# RetrievalQA Chain - deprecated!!
# 현재는 LCEL 방식 사용
%pip install -qU langchain langchainhub 


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_core.prompts import PromptTemplate

template = """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

Question: {question} 
Context: {context} 
Answer:"""

prompt = PromptTemplate.from_template(template)

In [4]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\nQuestion: {question} \nContext: {context} \nAnswer:")

In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 벡터 DB를 retriever 모드로 변환
retriever = database.as_retriever()

# 검색해 온 문서 조각을 긴 텍스트로 이어 붙여주는 헬퍼 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL 문법으로 RAG Chain 조립
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [8]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
final_answer = rag_chain.invoke(query)

print("🤖 AI 답변:")
print(final_answer)

🤖 AI 답변:
I don't know the specific income tax amount for an annual salary of 50 million won based on the given context.
